In [ ]:
!pip -q install -U datasets transformers scikit-learn pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from sklearn.model_selection import train_test_split

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [ ]:
import json
import requests
import pandas as pd

BASE_URL = (
    "https://huggingface.co/datasets/Coldog2333/tiage"
    "/resolve/main/"
)

def load_tiage_split(split):
    url = BASE_URL + f"{split}.json"

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    data = response.json()["dial_data"]["tiage"]
    return data

train_dialogues = load_tiage_split("train")
val_dialogues = load_tiage_split("validation")
test_dialogues = load_tiage_split("test")

print("Train dialogues:", len(train_dialogues))
print("Validation dialogues:", len(val_dialogues))
print("Test dialogues:", len(test_dialogues))

# Inspect the actual structure
print("\nFirst dialogue:")
print(json.dumps(train_dialogues[0], indent=2)[:3000])

Train dialogues: 300
Validation dialogues: 100
Test dialogues: 100

First dialogue:
{
  "dial_id": "tiage_dial_001",
  "turns": [
    {
      "da": "",
      "role": "user",
      "turn_id": 1,
      "utterance": "hey ! do you love cats ?",
      "topic_id": 0,
      "segmentation_label": 0
    },
    {
      "da": "",
      "role": "agent",
      "turn_id": 2,
      "utterance": "hey . . . i am a dog person , i have two",
      "topic_id": 0,
      "segmentation_label": 0
    },
    {
      "da": "",
      "role": "user",
      "turn_id": 3,
      "utterance": "ah that is cool , i have two cats and got a collection of 1000 hats for them !",
      "topic_id": 0,
      "segmentation_label": 0
    },
    {
      "da": "",
      "role": "agent",
      "turn_id": 4,
      "utterance": "wow ! ! ! that is a lot lol",
      "topic_id": 0,
      "segmentation_label": 1
    },
    {
      "da": "",
      "role": "user",
      "turn_id": 5,
      "utterance": "yeah , i have a weakness for cats a

In [ ]:
def create_examples(dialogues, history_turns=3):
    examples = []

    for dialogue in dialogues:
        turns = dialogue["turns"]

        for i in range(1, len(turns)):
            previous_topic = turns[i - 1]["topic_id"]
            current_topic = turns[i]["topic_id"]

            # 1 = topic shift, 0 = continuation
            label = int(current_topic != previous_topic)

            start = max(0, i - history_turns)

            history = "\n".join(
                f'{turns[j]["role"]}: {turns[j]["utterance"]}'
                for j in range(start, i)
            )

            current = turns[i]["utterance"]

            text = (
                f"Previous conversation:\n{history}\n\n"
                f"Current utterance:\n{current}"
            )

            examples.append({
                "dial_id": dialogue["dial_id"],
                "text": text,
                "label": label,
                "current_utterance": current
            })

    return pd.DataFrame(examples)


train_all = create_examples(train_dialogues)
val_all = create_examples(val_dialogues)

print("Training examples:", len(train_all))
print("Validation examples:", len(val_all))

print("\nTraining labels:")
print(train_all["label"].value_counts())

print("\nValidation labels:")
print(val_all["label"].value_counts())

Training examples: 4392
Validation examples: 1446

Training labels:
label
0    3505
1     887
Name: count, dtype: int64

Validation labels:
label
0    1135
1     311
Name: count, dtype: int64


In [ ]:
def sample_examples(df, n, seed=42):
    """Sample examples while preserving class proportions when possible."""
    if len(df) <= n:
        return df.reset_index(drop=True)

    if df["label"].nunique() == 2 and df["label"].value_counts().min() >= 2:
        sampled, _ = train_test_split(
            df,
            train_size=n,
            stratify=df["label"],
            random_state=seed
        )
        return sampled.reset_index(drop=True)

    return df.sample(n=n, random_state=seed).reset_index(drop=True)


train_df = sample_examples(train_all, 800, SEED)
val_df = sample_examples(val_all, 200, SEED)

print("\nSelected training examples:", len(train_df))
print("Selected validation examples:", len(val_df))

print("\nTraining distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

print("\nSample example:")
print("Text:\n", train_df.iloc[0]["text"])
print("Label:", train_df.iloc[0]["label"])


Selected training examples: 800
Selected validation examples: 200

Training distribution:
label
0    638
1    162
Name: count, dtype: int64

Validation distribution:
label
0    157
1     43
Name: count, dtype: int64

Sample example:
Text:
 Previous conversation:
user: hey , what are you planning for today ?

Current utterance:
oh not much , my name is heather and just looking to chat with people
Label: 0


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 192
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 2e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    dtype=torch.float32
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    eps=1e-8
)

criterion = torch.nn.CrossEntropyLoss()

class DialogueDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "label": self.labels[idx]
        }


def collate_batch(batch):
    texts = [item["text"] for item in batch]
    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long
    )

    encodings = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    encodings["labels"] = labels
    return encodings


train_loader = DataLoader(
    DialogueDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

val_loader = DataLoader(
    DialogueDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / max(len(train_loader), 1)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"| Training loss: {average_loss:.4f}"
    )

Using device: cuda


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.den

Epoch 1/100 | Training loss: 0.5763
Epoch 2/100 | Training loss: 0.5068
Epoch 3/100 | Training loss: 0.4987
Epoch 4/100 | Training loss: 0.4842
Epoch 5/100 | Training loss: 0.4476
Epoch 6/100 | Training loss: 0.3921
Epoch 7/100 | Training loss: 0.2920
Epoch 8/100 | Training loss: 0.2730
Epoch 9/100 | Training loss: 0.1080
Epoch 10/100 | Training loss: 0.0579
Epoch 11/100 | Training loss: 0.0800
Epoch 12/100 | Training loss: 0.0600
Epoch 13/100 | Training loss: 0.0782
Epoch 14/100 | Training loss: 0.0253
Epoch 15/100 | Training loss: 0.0080
Epoch 16/100 | Training loss: 0.0205
Epoch 17/100 | Training loss: 0.0405
Epoch 18/100 | Training loss: 0.0151
Epoch 19/100 | Training loss: 0.0104
Epoch 20/100 | Training loss: 0.0142
Epoch 21/100 | Training loss: 0.0025
Epoch 22/100 | Training loss: 0.0015
Epoch 23/100 | Training loss: 0.0011
Epoch 24/100 | Training loss: 0.0009
Epoch 25/100 | Training loss: 0.0007
Epoch 26/100 | Training loss: 0.0005
Epoch 27/100 | Training loss: 0.0005
Epoch 28/1

KeyboardInterrupt: 

In [ ]:
model.eval()

all_predictions = []
all_labels = []
all_probabilities = []

with torch.no_grad():
    for batch in val_loader:
        labels = batch["labels"].numpy()

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        outputs = model(**inputs)

        probabilities = torch.softmax(outputs.logits, dim=-1)
        predictions = torch.argmax(probabilities, dim=-1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels)

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )


print("\nAccuracy:", round(
    accuracy_score(all_labels, all_predictions), 4
))

print("Macro F1:", round(
    f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    ), 4
))

print("\nClassification report:")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1],
    target_names=["CONTINUATION", "TOPIC_SHIFT"],
    zero_division=0
))

print("Confusion matrix:")
print(
    confusion_matrix(
        all_labels,
        all_predictions,
        labels=[0, 1]
    )
)


Accuracy: 0.865
Macro F1: 0.7636

Classification report:
              precision    recall  f1-score   support

CONTINUATION       0.87      0.97      0.92       157
 TOPIC_SHIFT       0.81      0.49      0.61        43

    accuracy                           0.86       200
   macro avg       0.84      0.73      0.76       200
weighted avg       0.86      0.86      0.85       200

Confusion matrix:
[[152   5]
 [ 22  21]]


In [ ]:
import numpy as np

y_true = np.asarray(all_labels)
y_pred = np.asarray(all_predictions)

false_negatives = np.where(
    (y_true == 1) & (y_pred == 0)
)[0]

for idx in false_negatives[:10]:
    print(f"\nExample {idx}")
    print("Actual: TOPIC_SHIFT")
    print("Predicted: CONTINUATION")
    print(val_df.iloc[idx]["text"])
    print("-" * 80)


Example 2
Actual: TOPIC_SHIFT
Predicted: CONTINUATION
Previous conversation:
agent: i like kids , i would love to have a large family .
user: my students are my family .
agent: how sweet ! more teachers need to have that attitude .

Current utterance:
i am trying to teach them spanish and french so they are fluent like me .
--------------------------------------------------------------------------------

Example 8
Actual: TOPIC_SHIFT
Predicted: CONTINUATION
Previous conversation:
user: hey ! how are you doing ?
agent: great how are you ? i went to the park today .
user: i bet that was very relaxing . i wish i could leave the center

Current utterance:
what center are you at ? i am work at a center as a doc .
--------------------------------------------------------------------------------

Example 32
Actual: TOPIC_SHIFT
Predicted: CONTINUATION
Previous conversation:
agent: yes i do after all i am a engineer .
user: nice . my cousin is an engineer and hates video games . but i love donu

In [ ]:
import numpy as np

y_true = np.asarray(all_labels)
y_pred = np.asarray(all_predictions)

# Indices where the model predicted TOPIC_SHIFT
predicted_shifts = np.where(y_pred == 1)[0]

print(f"Total predicted topic shifts: {len(predicted_shifts)}")
print(f"Correct topic shifts: {(y_true[predicted_shifts] == 1).sum()}")
print(f"Incorrect topic shifts: {(y_true[predicted_shifts] == 0).sum()}")

for idx in predicted_shifts[:20]:
    actual_label = "TOPIC_SHIFT" if y_true[idx] == 1 else "CONTINUATION"

    print(f"\nExample {idx}")
    print(f"Actual: {actual_label}")
    print("Predicted: TOPIC_SHIFT")
    print(val_df.iloc[idx]["text"])
    print("-" * 80)

Total predicted topic shifts: 26
Correct topic shifts: 21
Incorrect topic shifts: 5

Example 45
Actual: CONTINUATION
Predicted: TOPIC_SHIFT
Previous conversation:
user: waylon jennings is my favorite ! next weekend is county wide bake sale . you coming ?
agent: i m in jersey . i m a runner . i have a marathon next weekend .
user: oh good luck ! the bake sale feels like a second home to me !

Current utterance:
i bet . what are you making ? i have a great pb cookie recipe .
--------------------------------------------------------------------------------

Example 46
Actual: CONTINUATION
Predicted: TOPIC_SHIFT
Previous conversation:
user: hey ! how are you doing ?

Current utterance:
great how are you ? i went to the park today .
--------------------------------------------------------------------------------

Example 47
Actual: CONTINUATION
Predicted: TOPIC_SHIFT
Previous conversation:
user: you know my fifth son keeps asking me for a dog . it is just too much .
agent: he can have mine .